In [ ]:
import json
from collections import Counter
import statistics
from itertools import combinations

with open('./data/raw/hatexplain/dataset.json', 'r') as f:
    data = json.load(f)

print(f"전체 포스트 수: {len(data)}")

전체 포스트 수: 20148


In [ ]:
def majority_label(annotators):
    """3명 어노테이터의 다수결 라벨"""
    labels = [a['label'] for a in annotators]
    return Counter(labels).most_common(1)[0][0]

def get_targets(annotators):
    """3명 어노테이터의 타겟 합집합 (None 제외)"""
    targets = []
    for a in annotators:
        for t in a['target']:
            if t != 'None':
                targets.append(t)
    return list(set(targets))

# hate/offensive만 필터링
ho = {}
for k, v in data.items():
    ml = majority_label(v['annotators'])
    if ml in ['hatespeech', 'offensive']:
        ho[k] = {
            **v,
            'ml': ml,
            'tl': [t.lower() for t in v['post_tokens']]  # 소문자화 토큰
        }

print(f"Hate/Offensive: {len(ho)}")
print(f"  hatespeech: {sum(1 for v in ho.values() if v['ml']=='hatespeech')}")
print(f"  offensive:  {sum(1 for v in ho.values() if v['ml']=='offensive')}")

Hate/Offensive: 11995
  hatespeech: 6234
  offensive:  5761


## 공통 Lexicon 정의
분석 전반에서 사용하는 수작업 lexicon들을 한곳에 정의.

In [8]:
# === Polarity Cue Lexicons ===

profanity = {
    'fuck','fucking','fucked','fucker','fuckers','fucks','motherfucker','motherfucking',
    'shit','shitty','bullshit','shits','shitting',
    'ass','asshole','assholes','damn','damned','dammit','goddamn',
    'bitch','bitches','bitching','bastard','bastards',
    'crap','crappy','piss','pissed','pissing',
    'hell','cunt','cunts','dick','dicks','prick','cock',
    'wtf','stfu','lmfao','smfh'
}

slurs = {
    'nigger','niggers','nigga','niggas','coon','coons','negro','negroes','darkie','niglet','niglets','sheboon',
    'kike','kikes','yid','yids','sheeny','heeb',
    'moslem','moslems','muzzie','muzzies','muzrat','muzrats','mudslime','mudslimes','raghead','towelhead','sandnigger',
    'faggot','faggots','fag','fags','dyke','dykes','homo','homos','tranny','trannies',
    'chink','chinks','spic','spics','wetback','wetbacks','gook','gooks','beaner','beaners',
    'whore','whores','slut','sluts','hoe','hoes','thot',
    'retard','retarded','retards'
}

neg_emotion = {
    'hate','hated','hates','hating','despise','despised','loathe','loathing','detest',
    'disgusting','disgust','disgusted','revolting','repulsive',
    'pathetic','horrible','terrible','awful','worst','horrendous','atrocious',
    'stupid','idiot','idiots','idiotic','dumb','moron','morons','imbecile',
    'ugly','hideous','filthy','nasty','gross','vile',
    'evil','wicked','scum','scumbag','trash','garbage',
    'worthless','useless','loser','losers','lame',
    'creepy','sick','twisted','degenerate','degenerates','subhuman'
}

intensifiers = {
    'very','really','so','extremely','absolutely','totally','completely','utterly',
    'fucking','freaking','damn','goddamn','bloody','insanely','incredibly',
    'super','mega','hella','af','asf'
}

violence_words = {
    'kill','killed','killing','murder','murdered','die','death','dead',
    'shoot','shot','hang','hanged','lynch','lynched','burn','stab','stabbed',
    'bomb','nuke','destroy','destroyed','exterminate','eliminate','eradicate',
    'gas','gassed','execute','executed','slaughter','massacre'
}

# 통합 strong cue set
all_cue = profanity | slurs | neg_emotion | violence_words

# === Target Reference Lexicons ===

target_ref_tokens = {
    'black','blacks','african','africans','colored',
    'jew','jews','jewish','hebrew','zionist','zionists','israel','israeli',
    'muslim','muslims','islam','islamic','quran','mosque',
    'women','woman','girl','girls','female','females','lady','feminist','feminists',
    'gay','gays','lesbian','lesbians','homosexual','homosexuals','lgbt','lgbtq','bisexual','transgender',
    'immigrant','immigrants','refugee','refugees','alien','aliens','migrant','migrants','illegal','illegals',
    'white','whites','caucasian','caucasians',
    'asian','asians','chinese','indian','pakistani',
    'arab','arabs','mexican','mexicans','hispanic','hispanics','latino','latina',
    'men','man','male','males','christian','christians',
}

# 타겟별 슬러/중립 사전
target_lexicons = {
    'African': {
        'slurs': {'nigger','niggers','nigga','niggas','coon','coons','negro','negroes','darkie','niglet','niglets','sheboon'},
        'neutral': {'black','blacks','african','africans','colored'}
    },
    'Jewish': {
        'slurs': {'kike','kikes','yid','yids','sheeny','heeb'},
        'neutral': {'jew','jews','jewish','hebrew','zionist','zionists','israel','israeli'}
    },
    'Islam': {
        'slurs': {'moslem','moslems','muzzie','muzzies','muzrat','muzrats','mudslime','mudslimes','raghead','towelhead','sandnigger'},
        'neutral': {'muslim','muslims','islam','islamic','quran','mosque','jihad','jihadi','sharia'}
    },
    'Women': {
        'slurs': {'bitch','bitches','slut','sluts','whore','whores','hoe','hoes','cunt','cunts','thot','feminazi'},
        'neutral': {'women','woman','girl','girls','female','females','lady','ladies','feminist','feminists'}
    },
    'Homosexual': {
        'slurs': {'faggot','faggots','fag','fags','dyke','dykes','homo','homos','tranny','trannies','queer','queers'},
        'neutral': {'gay','gays','lesbian','lesbians','homosexual','homosexuals','lgbtq','lgbt','bisexual','transgender'}
    },
    'Refugee': {
        'slurs': set(),
        'neutral': {'immigrant','immigrants','refugee','refugees','alien','aliens','migrant','migrants','illegal','illegals'}
    }
}

# === Stopwords ===
stopwords = {
    'the','a','an','is','was','are','were','be','been','being','have','has','had',
    'do','does','did','will','would','could','should','may','might','shall','can',
    'to','of','in','for','on','with','at','by','from','as','into','through','during',
    'and','but','or','if','while','that','this','these','those','i','me','my','we',
    'our','you','your','he','him','his','she','her','it','its','they','them','their',
    'not','no','all','just','so','than','very','about','up','get','got','like','one',
    '<user>','<number>','rt','amp','dont','im','ive','out','what','know','who','how',
    'when','where','why','can','than','there','been','more','some','only','even'
}

print("Lexicon 정의 완료")
print(f"  profanity: {len(profanity)}개, slurs: {len(slurs)}개")
print(f"  neg_emotion: {len(neg_emotion)}개, violence: {len(violence_words)}개")
print(f"  all_cue (통합): {len(all_cue)}개")
print(f"  target_ref: {len(target_ref_tokens)}개")

Lexicon 정의 완료
  profanity: 41개, slurs: 59개
  neg_emotion: 52개, violence: 30개
  all_cue (통합): 182개
  target_ref: 71개


---
## 1. 기본 EDA
### 1-1. 문장 길이 분포

In [9]:
# 전체 + 라벨별 길이 통계
for label in ['전체', 'hatespeech', 'offensive']:
    if label == '전체':
        lens = [len(v['post_tokens']) for v in ho.values()]
    else:
        lens = [len(v['post_tokens']) for v in ho.values() if v['ml']==label]
    print(f"\n[{label}] n={len(lens)}")
    print(f"  mean={statistics.mean(lens):.1f}, median={statistics.median(lens):.1f}, std={statistics.stdev(lens):.1f}")
    print(f"  min={min(lens)}, Q1={sorted(lens)[len(lens)//4]}, Q3={sorted(lens)[3*len(lens)//4]}, max={max(lens)}")

# 구간별 분포
print(f"\n{'구간':>8s} | {'hatespeech':>12s} | {'offensive':>12s} | {'합계':>12s}")
print(f"{'-'*8}-+-{'-'*12}-+-{'-'*12}-+-{'-'*12}")
bins = [(1,5),(6,10),(11,15),(16,20),(21,25),(26,30),(31,40),(41,50),(51,100),(101,200)]
h_n = sum(1 for v in ho.values() if v['ml']=='hatespeech')
o_n = sum(1 for v in ho.values() if v['ml']=='offensive')
for lo, hi in bins:
    h = sum(1 for v in ho.values() if v['ml']=='hatespeech' and lo <= len(v['post_tokens']) <= hi)
    o = sum(1 for v in ho.values() if v['ml']=='offensive' and lo <= len(v['post_tokens']) <= hi)
    t = h + o
    print(f"{lo:>3d}-{hi:<3d}  | {h:>5d} ({h/h_n*100:5.1f}%) | {o:>5d} ({o/o_n*100:5.1f}%) | {t:>5d} ({t/len(ho)*100:5.1f}%)")

# 플랫폼별
print("\n[플랫폼별 길이]")
for platform in ['twitter', 'gab']:
    lens = [len(v['post_tokens']) for k,v in ho.items() if platform in k]
    if lens:
        print(f"  {platform}: n={len(lens)}, mean={statistics.mean(lens):.1f}, median={statistics.median(lens):.1f}")

# 라벨 × 플랫폼
print("\n[라벨 × 플랫폼]")
for label in ['hatespeech','offensive']:
    for platform in ['twitter','gab']:
        lens = [len(v['post_tokens']) for k,v in ho.items() if v['ml']==label and platform in k]
        if lens:
            print(f"  {label}-{platform}: n={len(lens)}, mean={statistics.mean(lens):.1f}, median={statistics.median(lens):.1f}")


[전체] n=11995
  mean=23.6, median=21.0, std=13.9
  min=2, Q1=12, Q3=34, max=165

[hatespeech] n=6234
  mean=24.7, median=22.0, std=14.1
  min=2, Q1=13, Q3=36, max=165

[offensive] n=5761
  mean=22.5, median=20.0, std=13.6
  min=2, Q1=11, Q3=32, max=76

      구간 |   hatespeech |    offensive |           합계
---------+--------------+--------------+-------------
  1-5    |   328 (  5.3%) |   337 (  5.8%) |   665 (  5.5%)
  6-10   |   811 ( 13.0%) |   956 ( 16.6%) |  1767 ( 14.7%)
 11-15   |   870 ( 14.0%) |   938 ( 16.3%) |  1808 ( 15.1%)
 16-20   |   829 ( 13.3%) |   788 ( 13.7%) |  1617 ( 13.5%)
 21-25   |   727 ( 11.7%) |   689 ( 12.0%) |  1416 ( 11.8%)
 26-30   |   592 (  9.5%) |   484 (  8.4%) |  1076 (  9.0%)
 31-40   |   942 ( 15.1%) |   701 ( 12.2%) |  1643 ( 13.7%)
 41-50   |   973 ( 15.6%) |   756 ( 13.1%) |  1729 ( 14.4%)
 51-100  |   161 (  2.6%) |   112 (  1.9%) |   273 (  2.3%)
101-200  |     1 (  0.0%) |     0 (  0.0%) |     1 (  0.0%)

[플랫폼별 길이]
  twitter: n=3181, mean=15.2

### 1-2. 타깃 집단 분포

In [10]:
# 전체 타겟 빈도 (복수 타겟 허용)
target_all = Counter()
for v in ho.values():
    for t in get_targets(v['annotators']):
        target_all[t] += 1

total = len(ho)
print("[전체 타겟 빈도]")
for t, c in target_all.most_common():
    pct = c/total*100
    bar = '█' * int(pct/2)
    print(f"  {t:15s}: {c:5d} ({pct:5.1f}%) {bar}")

# 포스트당 타겟 수
print("\n[포스트당 타겟 수]")
target_count_dist = Counter()
for v in ho.values():
    target_count_dist[len(get_targets(v['annotators']))] += 1
for n in sorted(target_count_dist.keys()):
    c = target_count_dist[n]
    print(f"  {n}개: {c:5d} ({c/total*100:.1f}%)")

# 라벨별 타겟 분포
for label in ['hatespeech', 'offensive']:
    target_label = Counter()
    label_n = sum(1 for v in ho.values() if v['ml']==label)
    for v in ho.values():
        if v['ml'] == label:
            for t in get_targets(v['annotators']):
                target_label[t] += 1
    print(f"\n[{label} 타겟 분포] n={label_n}")
    for t, c in target_label.most_common(12):
        print(f"  {t:15s}: {c:5d} ({c/label_n*100:.1f}%)")

# 플랫폼별
print("\n[플랫폼별 타겟 — 상위 8]")
for platform in ['twitter', 'gab']:
    target_plat = Counter()
    plat_n = sum(1 for k in ho if platform in k)
    for k, v in ho.items():
        if platform in k:
            for t in get_targets(v['annotators']):
                target_plat[t] += 1
    print(f"\n  {platform} (n={plat_n}):")
    for t, c in target_plat.most_common(8):
        print(f"    {t:15s}: {c:5d} ({c/plat_n*100:.1f}%)")

# 타겟별 평균 문장 길이 (단일 타겟)
print("\n[타겟별 평균 길이 — 단일 타겟]")
target_lens = {}
for v in ho.values():
    targets = get_targets(v['annotators'])
    if len(targets) == 1:
        t = targets[0]
        target_lens.setdefault(t, []).append(len(v['post_tokens']))
for t in sorted(target_lens, key=lambda x: -len(target_lens[x])):
    lens = target_lens[t]
    if len(lens) >= 30:
        print(f"  {t:15s}: n={len(lens):4d}, mean={statistics.mean(lens):5.1f}, median={statistics.median(lens):5.1f}")

[전체 타겟 빈도]
  African        :  3724 ( 31.0%) ███████████████
  Women          :  2838 ( 23.7%) ███████████
  Other          :  2672 ( 22.3%) ███████████
  Islam          :  2542 ( 21.2%) ██████████
  Jewish         :  2228 ( 18.6%) █████████
  Homosexual     :  1916 ( 16.0%) ███████
  Arab           :  1544 ( 12.9%) ██████
  Caucasian      :  1096 (  9.1%) ████
  Refugee        :  1095 (  9.1%) ████
  Men            :  1084 (  9.0%) ████
  Hispanic       :   594 (  5.0%) ██
  Asian          :   525 (  4.4%) ██
  Minority       :   175 (  1.5%) 
  Christian      :   145 (  1.2%) 
  Disability     :   127 (  1.1%) 
  Heterosexual   :   120 (  1.0%) 
  Nonreligious   :    80 (  0.7%) 
  Indigenous     :    74 (  0.6%) 
  Economic       :    71 (  0.6%) 
  Indian         :    70 (  0.6%) 
  Hindu          :    44 (  0.4%) 
  Buddhism       :     9 (  0.1%) 
  Bisexual       :     7 (  0.1%) 
  Asexual        :     5 (  0.0%) 

[포스트당 타겟 수]
  0개:   203 (1.7%)
  1개:  5393 (45.0%)
  2개:  3645 

---
## 2. 라벨 구조 분석

In [11]:
# 일치도 분석
print("[일치도]")
for label in ['전체', 'hatespeech', 'offensive']:
    if label == '전체':
        subset = list(ho.values())
    else:
        subset = [v for v in ho.values() if v['ml']==label]
    n = len(subset)
    unan = sum(1 for v in subset if len(set(a['label'] for a in v['annotators']))==1)
    two = sum(1 for v in subset if len(set(a['label'] for a in v['annotators']))==2)
    three = sum(1 for v in subset if len(set(a['label'] for a in v['annotators']))==3)
    print(f"\n  {label} (n={n}):")
    print(f"    만장일치: {unan} ({unan/n*100:.1f}%)")
    print(f"    2:1:     {two} ({two/n*100:.1f}%)")
    print(f"    3-way:   {three} ({three/n*100:.1f}%)")

# 소수 의견 패턴
print("\n[소수 의견 패턴]")
for label in ['hatespeech', 'offensive']:
    dissent = Counter()
    for v in ho.values():
        if v['ml'] != label: continue
        for a in v['annotators']:
            if a['label'] != label:
                dissent[a['label']] += 1
    print(f"  {label}의 소수 의견: {dict(dissent)}")

# 타겟 어노테이션 일치도
print("\n[타겟 어노테이션 일치도]")
target_agree = Counter()
for v in ho.values():
    ann_targets = [frozenset(a['target']) for a in v['annotators']]
    if len(set(ann_targets)) == 1: target_agree['all_same'] += 1
    elif len(set(ann_targets)) == len(ann_targets): target_agree['all_different'] += 1
    else: target_agree['partial'] += 1
for k, c in target_agree.items():
    print(f"  {k:16s}: {c:5d} ({c/len(ho)*100:.1f}%)")

# 라벨 × 타겟 불일치 교차
print("\n[라벨 불일치 × 타겟 불일치 교차]")
cross = Counter()
for v in ho.values():
    la = 'agree' if len(set(a['label'] for a in v['annotators']))==1 else 'disagree'
    ta = 'agree' if len(set(frozenset(a['target']) for a in v['annotators']))==1 else 'disagree'
    cross[(la, ta)] += 1
for (la, ta), c in sorted(cross.items()):
    print(f"  label_{la} + target_{ta}: {c:5d} ({c/len(ho)*100:.1f}%)")

# 경계 포스트 길이
print("\n[경계 포스트 길이 비교]")
for label in ['hatespeech', 'offensive']:
    pure = [len(v['post_tokens']) for v in ho.values()
            if v['ml']==label and len(set(a['label'] for a in v['annotators']))==1]
    mixed = [len(v['post_tokens']) for v in ho.values()
             if v['ml']==label and len(set(a['label'] for a in v['annotators']))>1]
    print(f"  {label} 만장일치: mean={statistics.mean(pure):.1f} (n={len(pure)})")
    print(f"  {label} 불일치:   mean={statistics.mean(mixed):.1f} (n={len(mixed)})")

# normal 소수의견 포스트
has_normal = sum(1 for v in ho.values() if 'normal' in [a['label'] for a in v['annotators']])
print(f"\n[normal 소수의견 포함]: {has_normal}/{len(ho)} ({has_normal/len(ho)*100:.1f}%)")

[일치도]

  전체 (n=11995):
    만장일치: 4721 (39.4%)
    2:1:     6694 (55.8%)
    3-way:   580 (4.8%)

  hatespeech (n=6234):
    만장일치: 2960 (47.5%)
    2:1:     2975 (47.7%)
    3-way:   299 (4.8%)

  offensive (n=5761):
    만장일치: 1761 (30.6%)
    2:1:     3719 (64.6%)
    3-way:   281 (4.9%)

[소수 의견 패턴]
  hatespeech의 소수 의견: {'offensive': 2534, 'normal': 1039}
  offensive의 소수 의견: {'hatespeech': 1962, 'normal': 2319}

[타겟 어노테이션 일치도]
  all_same        :  4024 (33.5%)
  partial         :  5169 (43.1%)
  all_different   :  2802 (23.4%)

[라벨 불일치 × 타겟 불일치 교차]
  label_agree + target_agree:  2014 (16.8%)
  label_agree + target_disagree:  2707 (22.6%)
  label_disagree + target_agree:  2010 (16.8%)
  label_disagree + target_disagree:  5264 (43.9%)

[경계 포스트 길이 비교]
  hatespeech 만장일치: mean=24.0 (n=2960)
  hatespeech 불일치:   mean=25.3 (n=3274)
  offensive 만장일치: mean=20.2 (n=1761)
  offensive 불일치:   mean=23.6 (n=4000)

[normal 소수의견 포함]: 3358/11995 (28.0%)


---
## 3. Target 그룹 중심 분석
### 3-1. 그룹별 Toxicity Profile

In [12]:
top_targets = ['African','Jewish','Islam','Women','Homosexual','Arab','Caucasian','Refugee','Hispanic','Asian','Men','Other']

print(f"{'타겟':>12s} | {'n':>5s} | {'hate%':>6s} | {'off%':>6s} | {'unan%':>6s} | {'cue%':>6s} | {'len':>5s} | {'multi%':>6s} | {'rat%':>5s}")
print(f"{'-'*12}-+-{'-'*5}-+-{'-'*6}-+-{'-'*6}-+-{'-'*6}-+-{'-'*6}-+-{'-'*5}-+-{'-'*6}-+-{'-'*5}")

for t in top_targets:
    posts = [v for v in ho.values() if t in get_targets(v['annotators'])]
    n = len(posts)
    if n < 30: continue
    hate_pct = sum(1 for p in posts if p['ml']=='hatespeech') / n * 100
    off_pct = 100 - hate_pct
    unan_pct = sum(1 for p in posts if len(set(a['label'] for a in p['annotators']))==1) / n * 100
    cue_pct = sum(1 for p in posts if any(t in all_cue for t in p['tl'])) / n * 100
    mean_len = statistics.mean([len(p['post_tokens']) for p in posts])
    multi_pct = sum(1 for p in posts if len(get_targets(p['annotators'])) >= 2) / n * 100
    rat_pct = sum(1 for p in posts if p['rationales'] and any(any(r) for r in p['rationales'])) / n * 100
    print(f"{t:>12s} | {n:5d} | {hate_pct:5.1f}% | {off_pct:5.1f}% | {unan_pct:5.1f}% | {cue_pct:5.1f}% | {mean_len:5.1f} | {multi_pct:5.1f}% | {rat_pct:4.1f}%")

          타겟 |     n |  hate% |   off% |  unan% |   cue% |   len | multi% |  rat%
-------------+-------+--------+--------+--------+--------+-------+--------+------
     African |  3724 |  70.6% |  29.4% |  44.8% |  81.0% |  24.6 |  63.8% | 96.5%
      Jewish |  2228 |  73.2% |  26.8% |  48.2% |  76.8% |  26.2 |  62.7% | 95.7%
       Islam |  2542 |  64.8% |  35.2% |  34.6% |  68.1% |  27.3 |  77.0% | 94.7%
       Women |  2838 |  39.0% |  61.0% |  38.0% |  77.0% |  23.6 |  77.8% | 95.7%
  Homosexual |  1916 |  48.1% |  51.9% |  40.7% |  80.7% |  23.1 |  69.8% | 96.1%
        Arab |  1544 |  67.0% |  33.0% |  37.0% |  70.3% |  27.1 |  94.6% | 95.1%
   Caucasian |  1096 |  42.4% |  57.6% |  32.4% |  64.1% |  26.6 |  90.9% | 93.8%
     Refugee |  1095 |  46.1% |  53.9% |  26.4% |  48.9% |  30.1 |  76.1% | 90.8%
    Hispanic |   594 |  64.1% |  35.9% |  38.0% |  74.4% |  27.5 |  79.5% | 96.0%
       Asian |   525 |  57.0% |  43.0% |  30.7% |  50.9% |  25.3 |  73.1% | 93.9%
         Men |  

### 3-2. 그룹별 Implicitness

In [13]:
# 타겟 지칭 전략 (단일 타겟)
print("[타겟 지칭 전략 — 슬러 vs 중립]")
for t in ['African','Jewish','Islam','Women','Homosexual','Refugee']:
    lex = target_lexicons[t]
    posts = [v for v in ho.values() if get_targets(v['annotators'])==[t]]
    n = len(posts)
    if n < 30: continue
    slur_only = sum(1 for p in posts if set(p['tl'])&lex['slurs'] and not set(p['tl'])&lex['neutral'])
    neut_only = sum(1 for p in posts if set(p['tl'])&lex['neutral'] and not set(p['tl'])&lex['slurs'])
    both = sum(1 for p in posts if set(p['tl'])&lex['slurs'] and set(p['tl'])&lex['neutral'])
    neither = n - slur_only - neut_only - both
    print(f"\n  {t} (n={n}): slur={slur_only/n*100:.1f}%  neutral={neut_only/n*100:.1f}%  both={both/n*100:.1f}%  neither={neither/n*100:.1f}%")

# 라벨 판단 난이도
print("\n\n[라벨 판단 난이도]")
print(f"{'타겟':>12s} | {'만장일치%':>8s} | {'h↔o혼동%':>8s} | {'→normal%':>8s}")
for t in top_targets:
    posts = [v for v in ho.values() if t in get_targets(v['annotators'])]
    n = len(posts)
    if n < 50: continue
    unan = sum(1 for p in posts if len(set(a['label'] for a in p['annotators']))==1) / n * 100
    ho_c = sum(1 for p in posts if {'hatespeech','offensive'} == set(a['label'] for a in p['annotators'])) / n * 100
    norm_c = sum(1 for p in posts if 'normal' in [a['label'] for a in p['annotators']]) / n * 100
    print(f"{t:>12s} | {unan:7.1f}% | {ho_c:7.1f}% | {norm_c:7.1f}%")

# Rationale 특성
print("\n[타겟별 Rationale 특성]")
for t in top_targets:
    posts = [v for v in ho.values() if t in get_targets(v['annotators'])]
    coverages, span_lens = [], []
    for p in posts:
        if not p['rationales']: continue
        for r in p['rationales']:
            if any(r):
                coverages.append(sum(r)/len(r))
                span_lens.append(sum(r))
                break
    if coverages:
        print(f"  {t:12s}: coverage_med={statistics.median(coverages):.3f}, span_len_med={statistics.median(span_lens):.1f}")

[타겟 지칭 전략 — 슬러 vs 중립]

  African (n=1349): slur=75.5%  neutral=6.4%  both=6.0%  neither=12.1%

  Jewish (n=831): slur=56.9%  neutral=29.0%  both=9.6%  neither=4.5%

  Islam (n=585): slur=42.2%  neutral=41.0%  both=4.3%  neither=12.5%

  Women (n=629): slur=71.2%  neutral=18.6%  both=5.2%  neither=4.9%

  Homosexual (n=579): slur=69.1%  neutral=21.2%  both=6.7%  neither=2.9%

  Refugee (n=262): slur=0.0%  neutral=93.1%  both=0.0%  neither=6.9%


[라벨 판단 난이도]
          타겟 |    만장일치% |   h↔o혼동% | →normal%
     African |    44.8% |    35.5% |    19.7%
      Jewish |    48.2% |    32.8% |    19.1%
       Islam |    34.6% |    41.6% |    23.8%
       Women |    38.0% |    31.6% |    30.3%
  Homosexual |    40.7% |    37.2% |    22.1%
        Arab |    37.0% |    43.9% |    19.1%
   Caucasian |    32.4% |    34.9% |    32.7%
     Refugee |    26.4% |    35.6% |    38.0%
    Hispanic |    38.0% |    39.7% |    22.2%
       Asian |    30.7% |    36.2% |    33.1%
         Men |    30.9% |    35.3

---
## 4. Surface Cue 분석
### 4-1. 욕설/비속어 vs 비욕설

In [14]:
categories = {
    'profanity': profanity,
    'slurs': slurs,
    'neg_emotion': neg_emotion,
    'intensifiers': intensifiers,
    'violence': violence_words
}

for label in ['hatespeech', 'offensive', '전체']:
    subset = list(ho.values()) if label=='전체' else [v for v in ho.values() if v['ml']==label]
    n = len(subset)
    print(f"\n[{label}] n={n}")
    for cat_name, cat_set in categories.items():
        has = sum(1 for v in subset if set(v['tl']) & cat_set)
        token_counts = Counter()
        for v in subset:
            for t in v['tl']:
                if t in cat_set: token_counts[t] += 1
        top3 = ', '.join(f"{t}({c})" for t,c in token_counts.most_common(3))
        print(f"  {cat_name:15s}: {has:5d} ({has/n*100:5.1f}%)  top: {top3}")

# Cue 조합 패턴
print("\n[Cue 카테고리 조합]")
combo_counter = Counter()
for v in ho.values():
    toks = set(v['tl'])
    present = tuple(sorted(cn for cn, cs in categories.items() if toks & cs))
    combo_counter[present if present else ('none',)] += 1
for combo, c in combo_counter.most_common(10):
    print(f"  {'+'.join(combo):50s}: {c:5d} ({c/len(ho)*100:.1f}%)")


[hatespeech] n=6234
  profanity      :  1621 ( 26.0%)  top: fuck(426), fucking(395), shit(359)
  slurs          :  4501 ( 72.2%)  top: nigger(1721), kike(918), niggers(633)
  neg_emotion    :  1036 ( 16.6%)  top: hate(241), stupid(125), dumb(93)
  intensifiers   :  1076 ( 17.3%)  top: so(541), fucking(395), really(131)
  violence       :   706 ( 11.3%)  top: kill(167), dead(74), die(70)

[offensive] n=5761
  profanity      :  1565 ( 27.2%)  top: bitch(489), fucking(295), shit(269)
  slurs          :  2269 ( 39.4%)  top: retarded(644), faggot(256), retard(236)
  neg_emotion    :   789 ( 13.7%)  top: hate(228), trash(109), stupid(89)
  intensifiers   :  1084 ( 18.8%)  top: so(544), fucking(295), really(177)
  violence       :   319 (  5.5%)  top: kill(52), killed(44), dead(36)

[전체] n=11995
  profanity      :  3186 ( 26.6%)  top: fucking(690), fuck(679), bitch(677)
  slurs          :  6770 ( 56.4%)  top: nigger(1887), kike(982), retarded(820)
  neg_emotion    :  1825 ( 15.2%)  top: hate

### 4-2. Intensifier/감정어 — 위 결과의 intensifiers/neg_emotion 행 참조

### 4-3. 문장 기능별 패턴

In [15]:
def classify_speech_act(tokens):
    acts = []
    if any(tokens[i]=='you' and i+1<len(tokens) for i in range(len(tokens))):
        acts.append('DIRECT_ADDRESS')
    if 'they' in tokens or 'all' in tokens or 'every' in tokens:
        for i in range(len(tokens)-1):
            if tokens[i+1] in ('are','is','was','were'):
                acts.append('GROUP_STATEMENT'); break
    imperative_starts = {'go','get','stop','shut','kill','ban','deport','remove','die','hang','fuck','stfu','gtfo','leave'}
    if tokens and tokens[0] in imperative_starts:
        acts.append('IMPERATIVE')
    elif any(t in imperative_starts for t in tokens[:3]):
        acts.append('IMPERATIVE')
    text = ' '.join(tokens)
    if '?' in text or (tokens and tokens[0] in ('why','how','what','when','where','who','do','does','did','is','are','can')):
        acts.append('QUESTION')
    if any(t in ('wish','hope','should','deserve') for t in tokens) or 'need to' in text or 'want to' in text:
        acts.append('WISH_DESIRE')
    past_indicators = {'went','saw','told','said','found','heard','caught','got','came','happened','tried','started'}
    if sum(1 for t in tokens if t in past_indicators) >= 2:
        acts.append('NARRATIVE')
    if len(tokens) <= 6:
        acts.append('SHORT_LABEL')
    sarcasm = {'lol','lmao','haha','hahaha','rofl','imagine','sure','right','wow','oh'}
    if any(t in sarcasm for t in tokens):
        acts.append('MOCKERY_SARCASM')
    if ('hate' in tokens and ('i' in tokens or 'we' in tokens)) or (tokens and tokens[0]=='fuck'):
        acts.append('HATE_DECLARATION')
    if not acts:
        acts.append('OTHER')
    return acts

act_counter = {'hatespeech': Counter(), 'offensive': Counter()}
for v in ho.values():
    for a in classify_speech_act(v['tl']):
        act_counter[v['ml']][a] += 1

print(f"{'기능':>25s} | {'hate':>12s} | {'off':>12s} | {'diff':>8s}")
print(f"{'-'*25}-+-{'-'*12}-+-{'-'*12}-+-{'-'*8}")
h_n = sum(1 for v in ho.values() if v['ml']=='hatespeech')
o_n = sum(1 for v in ho.values() if v['ml']=='offensive')
all_acts = sorted(set(list(act_counter['hatespeech'])+list(act_counter['offensive'])),
                  key=lambda x: -(act_counter['hatespeech'].get(x,0)+act_counter['offensive'].get(x,0)))
for act in all_acts:
    h = act_counter['hatespeech'].get(act,0)
    o = act_counter['offensive'].get(act,0)
    hp, op = h/h_n*100, o/o_n*100
    print(f"{act:>25s} | {h:5d}({hp:5.1f}%) | {o:5d}({op:5.1f}%) | {hp-op:+6.1f}%p")

                       기능 |         hate |          off |     diff
--------------------------+--------------+--------------+---------
                    OTHER |  2642( 42.4%) |  2534( 44.0%) |   -1.6%p
           DIRECT_ADDRESS |  1496( 24.0%) |  1402( 24.3%) |   -0.3%p
          GROUP_STATEMENT |   965( 15.5%) |   822( 14.3%) |   +1.2%p
          MOCKERY_SARCASM |   593(  9.5%) |   581( 10.1%) |   -0.6%p
              SHORT_LABEL |   477(  7.7%) |   522(  9.1%) |   -1.4%p
              WISH_DESIRE |   589(  9.4%) |   366(  6.4%) |   +3.1%p
                 QUESTION |   381(  6.1%) |   325(  5.6%) |   +0.5%p
               IMPERATIVE |   246(  3.9%) |   161(  2.8%) |   +1.2%p
         HATE_DECLARATION |   207(  3.3%) |   169(  2.9%) |   +0.4%p
                NARRATIVE |    66(  1.1%) |    44(  0.8%) |   +0.3%p


---
## 5. 프레이밍 분석

In [16]:
def classify_framing(tokens):
    text = ' '.join(tokens)
    toks = set(tokens)
    frames = []
    
    dehuman = {'animal','animals','monkey','monkeys','ape','apes','cockroach','cockroaches',
               'vermin','pest','pests','rat','rats','dog','dogs','pig','pigs','subhuman',
               'savage','savages','parasite','parasites','infestation','infest','plague',
               'creature','creatures','beast','beasts','insect','filth','scum'}
    threat = {'kill','hang','gas','shoot','bomb','nuke','exterminate','execute','lynch',
              'murder','slaughter','purge','cleanse','eradicate','eliminate','burn','stab','genocide','holocaust'}
    conspiracy = {'control','controlling','destroy','destroying','ruin','ruining','invasion',
                 'invade','invading','replace','replacing','replacement','overthrow','infiltrate',
                 'conspire','conspiracy','plot','agenda','shill','shills','globalist','puppet',
                 'brainwash','propaganda','takeover','manipulate','manipulating'}
    purity = {'disgusting','disgust','filthy','dirty','unclean','impure','immoral','degenerate',
              'degenerates','pervert','perverts','perverted','abomination','sick','sickening',
              'vile','repulsive','revolting','sinful','unholy'}
    intellect = {'stupid','dumb','idiot','idiots','idiotic','moron','morons','imbecile',
                'brainless','uneducated','illiterate','ignorant','retarded','retard','braindead',
                'clueless','incompetent','dense'}
    sexual = {'rape','raped','raping','rapist','rapists','breed','breeding','molest','molested',
              'slut','sluts','whore','whores','hoe','hoes','thot','promiscuous','slutty'}
    criminal = {'crime','criminal','criminals','thug','thugs','gang','gangs','terrorist',
                'terrorists','terrorism','steal','stealing','theft','rob','robbery',
                'illegal','illegals','felon','pedophile','predator'}
    economic = {'welfare','leech','leeches','freeloader','freeloaders','parasite','parasites',
                'taxpayer','taxpayers','burden','lazy','handout','handouts','mooch'}
    religious = {'jihad','jihadi','sharia','infidel','heathen','godless','blasphemy',
                'satanic','satan','antichrist','crusade','islamist','radical islam'}
    
    if toks & dehuman: frames.append('DEHUMANIZATION')
    if toks & threat: frames.append('THREAT_VIOLENCE')
    if any(p in text for p in ['go back','get out','dont belong','not belong','not welcome',
            'our country','my country','send them','send back','deport','kick out','keep out']):
        frames.append('EXCLUSION')
    if toks & conspiracy: frames.append('CONSPIRACY')
    if toks & purity: frames.append('MORAL_DISGUST')
    if toks & intellect: frames.append('INTELLECTUAL_INFERIORITY')
    if toks & sexual: frames.append('SEXUAL_GENDERED')
    if toks & criminal: frames.append('CRIMINAL_DANGER')
    if toks & economic: frames.append('ECONOMIC_BURDEN')
    if toks & religious: frames.append('RELIGIOUS')
    if any(t in tokens for t in ('all','every','always','never')) and any(t in tokens for t in ('are','is','will','do','can','want','need','have')):
        frames.append('GENERALIZATION')
    if not frames: frames.append('NONE_DETECTED')
    return frames

# 전체 프레이밍 분포
frame_counter = {'hatespeech': Counter(), 'offensive': Counter()}
for v in ho.values():
    for f in classify_framing(v['tl']):
        frame_counter[v['ml']][f] += 1

print(f"{'프레이밍':>30s} | {'hate':>12s} | {'off':>12s} | {'diff':>8s}")
print(f"{'-'*30}-+-{'-'*12}-+-{'-'*12}-+-{'-'*8}")
all_frames = sorted(set(list(frame_counter['hatespeech'])+list(frame_counter['offensive'])),
                    key=lambda x: -(frame_counter['hatespeech'].get(x,0)+frame_counter['offensive'].get(x,0)))
for f in all_frames:
    h = frame_counter['hatespeech'].get(f,0); o = frame_counter['offensive'].get(f,0)
    hp, op = h/h_n*100, o/o_n*100
    print(f"{f:>30s} | {h:5d}({hp:5.1f}%) | {o:5d}({op:5.1f}%) | {hp-op:+6.1f}%p")

# 타겟별 지배적 프레이밍
print("\n[타겟별 지배적 프레이밍 — 단일 타겟]")
for t in ['African','Jewish','Islam','Women','Homosexual','Refugee']:
    tf = Counter()
    posts = [v for v in ho.values() if get_targets(v['annotators'])==[t]]
    n = len(posts)
    if n < 30: continue
    for p in posts:
        for f in classify_framing(p['tl']):
            if f != 'NONE_DETECTED': tf[f] += 1
    print(f"\n  {t} (n={n}):")
    for f, c in tf.most_common(5):
        print(f"    {f:30s}: {c:4d} ({c/n*100:.1f}%)")

                          프레이밍 |         hate |          off |     diff
-------------------------------+--------------+--------------+---------
                 NONE_DETECTED |  3508( 56.3%) |  3121( 54.2%) |   +2.1%p
      INTELLECTUAL_INFERIORITY |   465(  7.5%) |  1020( 17.7%) |  -10.2%p
                GENERALIZATION |   782( 12.5%) |   610( 10.6%) |   +2.0%p
               SEXUAL_GENDERED |   493(  7.9%) |   485(  8.4%) |   -0.5%p
               CRIMINAL_DANGER |   401(  6.4%) |   365(  6.3%) |   +0.1%p
               THREAT_VIOLENCE |   414(  6.6%) |   176(  3.1%) |   +3.6%p
                DEHUMANIZATION |   376(  6.0%) |   136(  2.4%) |   +3.7%p
                    CONSPIRACY |   259(  4.2%) |   182(  3.2%) |   +1.0%p
                 MORAL_DISGUST |   284(  4.6%) |   103(  1.8%) |   +2.8%p
                     EXCLUSION |   170(  2.7%) |    91(  1.6%) |   +1.1%p
                     RELIGIOUS |    91(  1.5%) |    62(  1.1%) |   +0.4%p
               ECONOMIC_BURDEN |    79(  1

---
## 6. Stereotype 분석

In [17]:
stereotype_lexicons = {
    'African': {
        'crime_violence': {'crime','criminal','criminals','murder','murdered','steal','stealing','robbery','assault','violent','violence','gang','gangs','thug','thugs','prison','jail','police','shooting'},
        'intelligence': {'stupid','dumb','iq','uneducated','illiterate','ignorant','retarded','braindead'},
        'laziness_welfare': {'lazy','welfare','handout','freeloader','work','working','unemployment'},
        'hypersexuality': {'breed','breeding','baby','babies','father','fatherless','rape'},
        'animal_savage': {'monkey','monkeys','ape','apes','animal','animals','savage','savages','gorilla','chimp','primitive'}
    },
    'Jewish': {
        'greed_money': {'money','rich','wealthy','greedy','greed','bank','banks','banker','gold','financial','billionaire'},
        'control_media': {'control','controlling','media','hollywood','news','propaganda','puppet','brainwash','manipulate'},
        'conspiracy': {'conspiracy','globalist','nwo','illuminati','rothschild','soros','zionist','deep state','cabal','plot'},
        'holocaust_denial': {'holocaust','holohoax','hoax','lie','lies','exaggerate'},
        'appearance': {'nose','rat','rats','ugly','hook'}
    },
    'Islam': {
        'terrorism': {'terrorist','terrorists','terrorism','bomb','bombing','jihad','jihadi','extremist','radical','isis'},
        'sexual_violence': {'rape','raped','raping','rapist','rapists','molest','grooming','pedophile'},
        'invasion': {'invasion','invade','invading','takeover','sharia','islamification','caliphate','outbreed'},
    },
    'Women': {
        'promiscuity': {'slut','sluts','whore','whores','hoe','hoes','thot','promiscuous','easy','gold digger'},
        'emotional_irrational': {'crazy','emotional','hysterical','irrational','dramatic','sensitive','overreact'},
        'appearance_value': {'ugly','fat','pretty','beautiful','hot','sexy','looks','weight','body'},
    },
    'Homosexual': {
        'disease_mental': {'disease','diseased','mental','illness','disorder','sick','sickness','abnormal','unnatural'},
        'pedophilia': {'pedophile','pedophiles','molest','molester','children','kids','groom','grooming','predator'},
        'religion_sin': {'sin','sinful','abomination','god','bible','hell','sodom','sodomy','repent'},
    },
    'Refugee': {
        'criminal': {'crime','criminal','criminals','rape','rapist','steal','drug','drugs','violence','violent','gang'},
        'economic_burden': {'welfare','tax','taxpayer','benefit','benefits','free','handout','cost','burden','leech','parasite'},
        'illegality': {'illegal','illegals','undocumented','border','deport','deportation','visa','overstay'},
    }
}

for target, cats in stereotype_lexicons.items():
    posts = [v for v in ho.values() if get_targets(v['annotators'])==[target]]
    n = len(posts)
    if n < 30: continue
    print(f"\n--- {target} (n={n}) ---")
    for cat_name, cat_words in cats.items():
        count = sum(1 for p in posts if set(p['tl']) & cat_words)
        pct = count/n*100
        word_freq = Counter(t for p in posts for t in p['tl'] if t in cat_words)
        top = ', '.join(f"{w}({c})" for w,c in word_freq.most_common(3))
        print(f"  {cat_name:25s}: {count:4d} ({pct:5.1f}%) → {top}")


--- African (n=1349) ---
  crime_violence           :   94 (  7.0%) → crime(30), violence(19), shooting(13)
  intelligence             :   78 (  5.8%) → stupid(33), dumb(20), retarded(15)
  laziness_welfare         :   34 (  2.5%) → work(14), welfare(12), working(6)
  hypersexuality           :   39 (  2.9%) → baby(13), babies(11), rape(10)
  animal_savage            :   53 (  3.9%) → animals(15), monkey(12), monkeys(9)

--- Jewish (n=831) ---
  greed_money              :   16 (  1.9%) → money(7), rich(6), billionaire(2)
  control_media            :   38 (  4.6%) → media(14), news(10), control(8)
  conspiracy               :   14 (  1.7%) → soros(7), zionist(5), conspiracy(1)
  holocaust_denial         :   38 (  4.6%) → holocaust(26), lie(12), lies(5)
  appearance               :   19 (  2.3%) → rat(11), ugly(3), nose(3)

--- Islam (n=585) ---
  terrorism                :   78 ( 13.3%) → terrorist(20), radical(14), jihadi(12)
  sexual_violence          :   45 (  7.7%) → rape(19), rape

---
## 7. Annotator Disagreement 분석

In [18]:
# 일치도별 특성 비교
unanimous = [v for v in ho.values() if len(set(a['label'] for a in v['annotators']))==1]
split_2_1 = [v for v in ho.values() if len(set(a['label'] for a in v['annotators']))==2]
three_way = [v for v in ho.values() if len(set(a['label'] for a in v['annotators']))==3]

for name, group in [('만장일치', unanimous), ('2:1', split_2_1), ('3-way', three_way)]:
    n = len(group)
    lens = [len(v['post_tokens']) for v in group]
    has_slur = sum(1 for v in group if set(v['tl']) & slurs) / n * 100
    tgt_counts = [len(get_targets(v['annotators'])) for v in group]
    print(f"\n{name} (n={n}):")
    print(f"  길이: mean={statistics.mean(lens):.1f}, 슬러포함: {has_slur:.1f}%, 타겟수: {statistics.mean(tgt_counts):.2f}")

# 불일치 유형
ho_conf = [v for v in split_2_1 if set(a['label'] for a in v['annotators'])=={'hatespeech','offensive'}]
hn_conf = [v for v in split_2_1 if 'normal' in [a['label'] for a in v['annotators']]]
print(f"\nhate↔off 혼동: {len(ho_conf)}건")
print(f"→normal 혼동:  {len(hn_conf)}건")

for name, group in [('hate↔off', ho_conf), ('→normal', hn_conf)]:
    n = len(group)
    if n == 0: continue
    slur_pct = sum(1 for v in group if set(v['tl']) & slurs) / n * 100
    mean_len = statistics.mean([len(v['post_tokens']) for v in group])
    print(f"  {name}: 슬러={slur_pct:.1f}%, 길이={mean_len:.1f}")

# normal 투표 있음 vs 없음
print("\n[normal 소수의견 특성]")
normal_vote = [v for v in ho.values() if 'normal' in [a['label'] for a in v['annotators']]]
no_normal = [v for v in ho.values() if 'normal' not in [a['label'] for a in v['annotators']]]
for name, group in [('normal투표 있음', normal_vote), ('normal투표 없음', no_normal)]:
    n = len(group)
    slur_pct = sum(1 for v in group if set(v['tl']) & slurs) / n * 100
    print(f"  {name} (n={n}): 슬러포함={slur_pct:.1f}%")


만장일치 (n=4721):
  길이: mean=22.6, 슬러포함: 71.4%, 타겟수: 1.88

2:1 (n=6694):
  길이: mean=24.2, 슬러포함: 48.7%, 타겟수: 1.91

3-way (n=580):
  길이: mean=26.1, 슬러포함: 23.3%, 타겟수: 1.88

hate↔off 혼동: 3916건
→normal 혼동:  2778건
  hate↔off: 슬러=58.3%, 길이=24.7
  →normal: 슬러=35.2%, 길이=23.5

[normal 소수의견 특성]
  normal투표 있음 (n=3358): 슬러포함=33.1%
  normal투표 없음 (n=8637): 슬러포함=65.5%


---
## 8. 어휘/구문 분석

In [19]:
# 8-1. 집단 일반화 vs 개인 지시
def classify_reference_scope(tokens):
    group_markers = {'all','every','always','never','these','those','them','they','their','typical','most','majority'}
    indiv_markers = {'this','that','he','she','him','her','you','your','one'}
    has_group = any(t in group_markers for t in tokens)
    has_indiv = any(t in indiv_markers for t in tokens)
    if has_group and not has_indiv: return 'GROUP_ONLY'
    elif has_indiv and not has_group: return 'INDIVIDUAL_ONLY'
    elif has_group and has_indiv: return 'MIXED'
    else: return 'UNCLEAR'

print("[집단 일반화 vs 개인 지시]")
for label in ['hatespeech', 'offensive']:
    scope = Counter()
    for v in ho.values():
        if v['ml'] == label:
            scope[classify_reference_scope(v['tl'])] += 1
    n = sum(scope.values())
    print(f"  {label}: " + ', '.join(f"{k}={v/n*100:.1f}%" for k,v in scope.most_common()))

# 8-2. 어휘 다양성 (TTR)
print("\n[어휘 다양성]")
for label in ['hatespeech', 'offensive']:
    all_tokens = [t for v in ho.values() if v['ml']==label for t in v['tl']]
    print(f"  {label}: total={len(all_tokens)}, unique={len(set(all_tokens))}, TTR={len(set(all_tokens))/len(all_tokens):.4f}")

# 8-3. 대명사 패턴
print("\n[대명사 사용]")
pronouns = {
    '1st_sing': {'i','me','my','mine','myself'},
    '1st_plur': {'we','us','our','ours','ourselves'},
    '2nd': {'you','your','yours','yourself'},
    '3rd_sing': {'he','him','his','she','her','hers','it','its'},
    '3rd_plur': {'they','them','their','theirs','themselves'}
}
for label in ['hatespeech', 'offensive']:
    n = sum(1 for v in ho.values() if v['ml']==label)
    print(f"  {label}:")
    for ptype, pset in pronouns.items():
        c = sum(1 for v in ho.values() if v['ml']==label and set(v['tl']) & pset)
        print(f"    {ptype:10s}: {c/n*100:.1f}%")

# 8-4. Cue-Target 토큰 거리
print("\n[Cue-Target 토큰 거리]")
distances = []
for v in ho.values():
    cue_pos = [i for i, t in enumerate(v['tl']) if t in all_cue]
    tgt_pos = [i for i, t in enumerate(v['tl']) if t in target_ref_tokens]
    if cue_pos and tgt_pos:
        distances.append(min(abs(c-t) for c in cue_pos for t in tgt_pos))

print(f"  공존 포스트: {len(distances)}/{len(ho)} ({len(distances)/len(ho)*100:.1f}%)")
print(f"  mean={statistics.mean(distances):.2f}, median={statistics.median(distances):.1f}")
dist_b = Counter()
for d in distances:
    if d <= 1: dist_b['0-1'] += 1
    elif d <= 3: dist_b['2-3'] += 1
    elif d <= 5: dist_b['4-5'] += 1
    elif d <= 10: dist_b['6-10'] += 1
    else: dist_b['11+'] += 1
for b in ['0-1','2-3','4-5','6-10','11+']:
    print(f"  {b:6s}: {dist_b[b]/len(distances)*100:.1f}%")

[집단 일반화 vs 개인 지시]
  hatespeech: INDIVIDUAL_ONLY=35.9%, UNCLEAR=25.9%, MIXED=23.0%, GROUP_ONLY=15.2%
  offensive: INDIVIDUAL_ONLY=36.2%, UNCLEAR=27.7%, MIXED=21.5%, GROUP_ONLY=14.6%

[어휘 다양성]
  hatespeech: total=153674, unique=15020, TTR=0.0977
  offensive: total=129824, unique=13750, TTR=0.1059

[대명사 사용]
  hatespeech:
    1st_sing  : 31.5%
    1st_plur  : 13.1%
    2nd       : 27.7%
    3rd_sing  : 33.3%
    3rd_plur  : 23.6%
  offensive:
    1st_sing  : 31.4%
    1st_plur  : 10.4%
    2nd       : 27.8%
    3rd_sing  : 31.8%
    3rd_plur  : 22.0%

[Cue-Target 토큰 거리]
  공존 포스트: 3353/11995 (28.0%)
  mean=5.78, median=3.0
  0-1   : 26.3%
  2-3   : 25.2%
  4-5   : 13.7%
  6-10  : 18.6%
  11+   : 16.2%


---
## 9. 임베딩/군집 분석

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np

texts = [' '.join(v['tl']) for v in ho.values()]
labels = [v['ml'] for v in ho.values()]
ho_list = list(ho.values())

# TF-IDF → SVD
tfidf = TfidfVectorizer(max_features=5000, min_df=5, max_df=0.8)
X = tfidf.fit_transform(texts)
svd = TruncatedSVD(n_components=50, random_state=42)
X_reduced = svd.fit_transform(X)
print(f"TF-IDF: {X.shape}, SVD explained variance: {svd.explained_variance_ratio_.sum():.3f}")

# K-Means
for k in [5, 8, 10]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    cl = km.fit_predict(X_reduced)
    sil = silhouette_score(X_reduced, cl, sample_size=5000, random_state=42)
    print(f"K={k}: silhouette={sil:.3f}")

# K=8 상세
km = KMeans(n_clusters=8, random_state=42, n_init=10)
cluster_labels = km.fit_predict(X_reduced)
feature_names = tfidf.get_feature_names_out()

print("\n[K=8 클러스터 상세]")
for c in range(8):
    mask = cluster_labels == c
    n = mask.sum()
    c_labels = [labels[i] for i in range(len(labels)) if mask[i]]
    hate_pct = sum(1 for l in c_labels if l=='hatespeech') / n * 100
    c_targets = Counter()
    for i in range(len(ho_list)):
        if mask[i]:
            for t in get_targets(ho_list[i]['annotators']): c_targets[t] += 1
    top_tgt = ', '.join(f"{t}({ct/n*100:.0f}%)" for t,ct in c_targets.most_common(3))
    cluster_tfidf = np.asarray(X[mask].mean(axis=0)).flatten()
    top_terms = ', '.join(feature_names[i] for i in cluster_tfidf.argsort()[-6:][::-1])
    c_lens = [len(ho_list[i]['post_tokens']) for i in range(len(ho_list)) if mask[i]]
    print(f"\n  C{c} (n={n}, {n/len(ho_list)*100:.1f}%): hate={hate_pct:.1f}%, len={statistics.mean(c_lens):.1f}")
    print(f"    targets: {top_tgt}")
    print(f"    terms: {top_terms}")

TF-IDF: (11995, 4460), SVD explained variance: 0.176
K=5: silhouette=0.019
K=8: silhouette=0.030
K=10: silhouette=0.033

[K=8 클러스터 상세]

  C0 (n=340, 2.8%): hate=59.1%, len=16.3
    targets: African(39%), Women(30%), Homosexual(23%)
    terms: fuck, you, the, nigger, and, bitch

  C1 (n=1738, 14.5%): hate=52.4%, len=32.6
    targets: Islam(30%), African(30%), Women(25%)
    terms: they, are, to, not, and, the

  C2 (n=719, 6.0%): hate=52.9%, len=27.8
    targets: African(32%), Islam(27%), Other(24%)
    terms: number, the, in, of, to, and

  C3 (n=2424, 20.2%): hate=54.4%, len=31.0
    targets: Islam(30%), African(29%), Jewish(25%)
    terms: the, of, and, to, is, in

  C4 (n=3527, 29.4%): hate=46.6%, len=18.4
    targets: Women(29%), Other(24%), African(22%)
    terms: is, to, kike, and, this, that

  C5 (n=929, 7.7%): hate=90.5%, len=16.1
    targets: African(94%), Women(14%), Arab(11%)
    terms: nigger, the, is, sand, this, to

  C6 (n=1326, 11.1%): hate=50.8%, len=26.0
    targets:

---
## 10. 상관/회귀/예측 분석

In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Feature 추출
feature_names_lr = ['length','has_slur','has_profanity','has_violence','has_neg_emotion',
                    'has_conspiracy','slur_count','profanity_count','target_count',
                    'platform_gab','has_question','has_1st_person','has_2nd_person','has_generalization']

conspiracy_set = {'control','destroy','ruin','invasion','invade','replace','conspiracy','plot','agenda','shill','globalist','puppet','brainwash','propaganda'}

features_list = []
for k, v in ho.items():
    toks = set(v['tl'])
    features_list.append({
        'ml': v['ml'],
        'length': len(v['post_tokens']),
        'has_slur': 1 if toks & slurs else 0,
        'has_profanity': 1 if toks & profanity else 0,
        'has_violence': 1 if toks & violence_words else 0,
        'has_neg_emotion': 1 if toks & neg_emotion else 0,
        'has_conspiracy': 1 if toks & conspiracy_set else 0,
        'slur_count': sum(1 for t in v['tl'] if t in slurs),
        'profanity_count': sum(1 for t in v['tl'] if t in profanity),
        'target_count': len(get_targets(v['annotators'])),
        'platform_gab': 1 if '_gab' in k else 0,
        'has_question': 1 if '?' in ' '.join(v['tl']) else 0,
        'has_1st_person': 1 if toks & {'i','me','my','we','us','our'} else 0,
        'has_2nd_person': 1 if toks & {'you','your','yours'} else 0,
        'has_generalization': 1 if toks & {'all','every','always','never'} else 0,
        'unanimous': 1 if len(set(a['label'] for a in v['annotators']))==1 else 0,
        'has_normal_vote': 1 if 'normal' in [a['label'] for a in v['annotators']] else 0,
    })

X_feat = np.array([[item[f] for f in feature_names_lr] for item in features_list])
y_hate = np.array([1 if item['ml']=='hatespeech' else 0 for item in features_list])

# 10-1. Feature별 hate 비율 차이
print("[Feature별 hate 비율 차이]")
for feat in feature_names_lr:
    if feat in ('length','slur_count','profanity_count','target_count'): continue
    vals = np.array([item[feat] for item in features_list])
    h1 = y_hate[vals==1].mean()*100 if vals.sum()>0 else 0
    h0 = y_hate[vals==0].mean()*100 if (vals==0).sum()>0 else 0
    print(f"  {feat:>25s}: =1→{h1:.1f}%  =0→{h0:.1f}%  diff={h1-h0:+.1f}%p")

# 10-2. Hate vs Offensive 예측
lr = LogisticRegression(max_iter=1000, random_state=42)
scores = cross_val_score(lr, X_feat, y_hate, cv=5, scoring='accuracy')
print(f"\n[Hate vs Off 예측] 5-fold CV: {scores.mean():.3f} (±{scores.std():.3f}), baseline: {max(y_hate.mean(),1-y_hate.mean()):.3f}")

lr.fit(X_feat, y_hate)
print("\n[Feature Importance]")
sorted_idx = np.argsort(np.abs(lr.coef_[0]))[::-1]
for idx in sorted_idx:
    d = '→ hate' if lr.coef_[0][idx] > 0 else '→ off'
    print(f"  {feature_names_lr[idx]:>25s}: {lr.coef_[0][idx]:+.4f} ({d})")

# 10-3. 불일치 예측
y_disagree = np.array([1 - item['unanimous'] for item in features_list])
lr2 = LogisticRegression(max_iter=1000, random_state=42)
scores2 = cross_val_score(lr2, X_feat, y_disagree, cv=5, scoring='accuracy')
print(f"\n[불일치 예측] 5-fold CV: {scores2.mean():.3f}, baseline: {max(y_disagree.mean(),1-y_disagree.mean()):.3f}")

lr2.fit(X_feat, y_disagree)
print("  Top features:")
for idx in np.argsort(np.abs(lr2.coef_[0]))[::-1][:5]:
    d = '→ 불일치' if lr2.coef_[0][idx] > 0 else '→ 일치'
    print(f"    {feature_names_lr[idx]:>25s}: {lr2.coef_[0][idx]:+.4f} ({d})")

# 10-4. Normal 소수의견 예측
y_normal = np.array([item['has_normal_vote'] for item in features_list])
lr3 = LogisticRegression(max_iter=1000, random_state=42)
scores3 = cross_val_score(lr3, X_feat, y_normal, cv=5, scoring='accuracy')
print(f"\n[Normal 투표 예측] 5-fold CV: {scores3.mean():.3f}")

lr3.fit(X_feat, y_normal)
print("  Top features:")
for idx in np.argsort(np.abs(lr3.coef_[0]))[::-1][:5]:
    d = '→ normal↑' if lr3.coef_[0][idx] > 0 else '→ normal↓'
    print(f"    {feature_names_lr[idx]:>25s}: {lr3.coef_[0][idx]:+.4f} ({d})")

[Feature별 hate 비율 차이]
                   has_slur: =1→66.5%  =0→33.2%  diff=+33.3%p
              has_profanity: =1→50.9%  =0→52.4%  diff=-1.5%p
               has_violence: =1→68.9%  =0→50.4%  diff=+18.5%p
            has_neg_emotion: =1→56.8%  =0→51.1%  diff=+5.7%p
             has_conspiracy: =1→57.5%  =0→51.8%  diff=+5.7%p
               platform_gab: =1→61.8%  =0→24.8%  diff=+37.1%p
               has_question: =1→0.0%  =0→52.0%  diff=-52.0%p
             has_1st_person: =1→53.2%  =0→51.2%  diff=+2.0%p
             has_2nd_person: =1→51.7%  =0→52.1%  diff=-0.3%p
         has_generalization: =1→54.7%  =0→51.5%  diff=+3.3%p

[Hate vs Off 예측] 5-fold CV: 0.711 (±0.112), baseline: 0.520

[Feature Importance]
               platform_gab: +1.5085 (→ hate)
                   has_slur: +0.9679 (→ hate)
               has_violence: +0.7899 (→ hate)
            has_neg_emotion: +0.3656 (→ hate)
                 slur_count: +0.3134 (→ hate)
             has_1st_person: +0.1158 (→ hate)
      

---
## Cue 없는 Hate vs Cue 있는 Hate

In [22]:
hate_posts = [v for v in ho.values() if v['ml']=='hatespeech']
with_cue = [v for v in hate_posts if any(t in all_cue for t in v['tl'])]
without_cue = [v for v in hate_posts if not any(t in all_cue for t in v['tl'])]

print(f"Cue 있음: {len(with_cue)} ({len(with_cue)/len(hate_posts)*100:.1f}%)")
print(f"Cue 없음: {len(without_cue)} ({len(without_cue)/len(hate_posts)*100:.1f}%)")

# 기본 특성
wc_lens = [len(v['post_tokens']) for v in with_cue]
wo_lens = [len(v['post_tokens']) for v in without_cue]
wc_unan = sum(1 for v in with_cue if len(set(a['label'] for a in v['annotators']))==1) / len(with_cue) * 100
wo_unan = sum(1 for v in without_cue if len(set(a['label'] for a in v['annotators']))==1) / len(without_cue) * 100
wc_norm = sum(1 for v in with_cue if 'normal' in [a['label'] for a in v['annotators']]) / len(with_cue) * 100
wo_norm = sum(1 for v in without_cue if 'normal' in [a['label'] for a in v['annotators']]) / len(without_cue) * 100

print(f"\n{'':30s} | {'CUE있음':>10s} | {'CUE없음':>10s}")
print(f"{'평균길이':>30s} | {statistics.mean(wc_lens):10.1f} | {statistics.mean(wo_lens):10.1f}")
print(f"{'만장일치%':>30s} | {wc_unan:9.1f}% | {wo_unan:9.1f}%")
print(f"{'normal소수의견%':>30s} | {wc_norm:9.1f}% | {wo_norm:9.1f}%")

# 타겟 분포
print("\n[타겟 분포]")
wc_t = Counter(t for v in with_cue for t in get_targets(v['annotators']))
wo_t = Counter(t for v in without_cue for t in get_targets(v['annotators']))
for t in ['African','Islam','Jewish','Refugee','Homosexual','Asian']:
    wc_pct = wc_t.get(t,0)/len(with_cue)*100
    wo_pct = wo_t.get(t,0)/len(without_cue)*100
    print(f"  {t:15s}: cue있음={wc_pct:.1f}%  cue없음={wo_pct:.1f}%  diff={wo_pct-wc_pct:+.1f}%p")

# 상위 어휘
print("\n[Cue 없는 hate 상위 토큰]")
wo_vocab = Counter(t for v in without_cue for t in v['tl'] if t not in stopwords and t not in all_cue and len(t)>1)
for tok, c in wo_vocab.most_common(20):
    print(f"  {tok:20s}: {c:4d} ({c/len(without_cue)*100:.1f}%)")

Cue 있음: 5255 (84.3%)
Cue 없음: 979 (15.7%)

                               |      CUE있음 |      CUE없음
                          평균길이 |       24.3 |       26.8
                         만장일치% |      52.0% |      23.3%
                   normal소수의견% |      13.5% |      33.8%

[타겟 분포]
  African        : cue있음=44.6%  cue없음=29.3%  diff=-15.3%p
  Islam          : cue있음=24.8%  cue없음=35.4%  diff=+10.7%p
  Jewish         : cue있음=26.9%  cue없음=22.0%  diff=-5.0%p
  Refugee        : cue있음=6.6%  cue없음=16.2%  diff=+9.7%p
  Homosexual     : cue있음=16.0%  cue없음=8.3%  diff=-7.7%p
  Asian          : cue있음=3.6%  cue없음=11.2%  diff=+7.6%p

[Cue 없는 hate 상위 토큰]
  white               :  215 (22.0%)
  jews                :  189 (19.3%)
  women               :  107 (10.9%)
  muslim              :  102 (10.4%)
  people              :   99 (10.1%)
  ghetto              :   71 (7.3%)
  blacks              :   61 (6.2%)
  raped               :   57 (5.8%)
  want                :   57 (5.8%)
  islam               :   56 (